
# Clustering algorithm and number of habitats

**Background.** The ``fit`` stage clusters the pooled supervoxels of the
training patients into habitats. Two choices are made there: the
clustering **algorithm** (k-means or a Gaussian mixture) and the
**number of habitats** K. HABIT tries every K in a range and keeps the
one a validation criterion prefers. Different criteria measure
different things, so on the same data they can pick different Ks.

**Purpose.** Starting from the analysis of
:doc:`/auto_examples/01_building_habitat_maps/plot_01_two_step_spec`, you will change
ONLY the ``fit`` stage, on the SAME pooled, binned supervoxels:

1. every k-means criterion, then every Gaussian-mixture criterion, and
   the K each one picks, with the score curves drawn by
   :func:`~habit.viz.plot_cluster_validation_from_report`;
2. the random seed (0, 1, 2) of the whole analysis: the K picked and the
   agreement of the habitat maps between seeds.

**When to use.** Before you freeze the habitat count of a study, and
whenever you have to justify "why K habitats" in a methods section.

**Key terms.** Full definitions are on :doc:`/tutorial/concepts`.

* **validation criterion** -- a score computed for every candidate K.
  k-means: ``elbow`` (knee of the within-cluster sum of squares,
  alias ``kneedle``; ``inertia`` is the same curve), ``silhouette`` and
  ``calinski_harabasz`` (higher is better), ``davies_bouldin`` (lower is
  better), ``gap`` (gap statistic, higher is better). Gaussian mixture:
  ``bic`` / ``aic`` (information criteria, lower is better),
  ``bic_elbow`` (knee of the BIC curve), plus the same four
  label-based scores.
* **vote** -- ``validation=[...]`` with several criteria: each criterion
  casts one vote and the most voted K is kept.
* **adjusted Rand index (ARI)** -- agreement of two label maps over the
  same ROI voxels (1 = same partition, about 0 = chance). It ignores
  label numbers, so it is safe when two fits number the same region
  differently.

The criteria comparison reuses the pooled, binned supervoxels of one
``Study`` run and refits only the fitter, so every criterion sees
exactly the same rows. The page checks first that refitting the
reference fitter on those rows reproduces the ``Study`` maps.

<div class="alert alert-info"><h4>Note</h4><p>Two training patients (60 pooled supervoxels) are a demo, not a
   study. With so few rows the criteria disagree easily; the point of
   the page is how to run and report the comparison.</p></div>


## The reference analysis
The Spec of the complete-analysis page, with only the ``volume``
quantify stage (the other feature families are not needed here).
``make_spec`` exists so the seed can be changed later without
touching anything else.
sphinx_gallery_thumbnail_number = 1



In [ ]:
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import adjusted_rand_score

from habit.contracts import cohort_from_directory
from habit.datasets import fetch_demo
from habit.habitat_model import GmmHabitatModelFitter, KMeansHabitatModelFitter
from habit.execution import backend_from_policy
from habit.recipes import Study
from habit.spec import HabitatSpec, Spec, Stage
from habit.spec.policy import RunPolicy
from habit.viz import plot_cluster_validation_from_report, plot_habitat_overlay

# Change DATA / MODALITIES / ROI to your preprocessed layout.
DATA = fetch_demo()
MODALITIES = ("pre_contrast", "LAP", "PVP", "delay_3min")
ROI = "LAP"
cohort = cohort_from_directory(DATA, modalities=MODALITIES, roi=ROI)
train = cohort[:2]
Path("out").mkdir(exist_ok=True)


def make_spec(seed: int) -> HabitatSpec:
    """
    Build the reference two-step Spec with a given random seed.

    Args:
        seed: Value of ``HabitatSpec.random_seed``; it seeds the k-means
            supervoxels and the k-means habitat fit.

    Returns:
        The HabitatSpec.
    """
    return HabitatSpec(
        name="clustering_choice_reference",
        stages=(
            Stage("extract", Spec("raw", {"modalities": list(MODALITIES), "roi": ROI})),
            Stage("preprocess", Spec("winsorize", {"winsor_limits": [0.01, 0.01]})),
            Stage("preprocess2", Spec("zscore")),
            Stage("partition", Spec("kmeans", {"n_supervoxels": 30})),
            Stage("pool", Spec("pool")),

            # The stage this page varies. A Gaussian mixture would be e.g.
            # Spec("gmm", {"min_habitats": 2, "max_habitats": 10, "validation": "bic", "n_init": 10}).
            Stage("fit", Spec("kmeans", {"min_habitats": 2, "max_habitats": 10, "validation": "elbow", "n_init": 10})),
            Stage("assign", Spec("nearest_centroid")),
            Stage("volume", Spec("volume")),
        ),
        random_seed=seed,
    )


result = Study(make_spec(0)).fit_predict(train)
print(result.habitat_model.summary())
# ``result.units`` are the supervoxels AFTER the cohort-level binning:
# exactly the rows the fitter clustered.
units = result.units
print("pooled supervoxels:", sum(one.feature_frame().shape[0] for one in units))

## Check: refitting the reference fitter reproduces Study
Same rows, same settings, same seed. If the labels are identical, the
criteria comparison below really compares criteria on the reference
data and nothing else.



In [ ]:
reference_fitter = KMeansHabitatModelFitter(min_habitats=2, max_habitats=10, validation="elbow", n_init=10)
reference_fitter.set_random_state(0)
reference_model = reference_fitter.fit(units, cohort=train)
reference_maps = [reference_model.assigner("nearest_centroid")(one) for one in units]
same = [np.array_equal(a.label_array, b.label_array) for a, b in zip(reference_maps, result.habitat_maps)]
print("K:", reference_model.n_habitats, "| labels identical to Study:", same)


def ari(maps_a: List[object], maps_b: List[object]) -> List[float]:
    """
    Adjusted Rand index between two lists of habitat maps, per patient.

    Args:
        maps_a: HabitatMap list.
        maps_b: HabitatMap list for the same patients, same order.

    Returns:
        One ARI (float) per patient over the ROI voxels (label > 0).
    """
    scores = []
    for map_a, map_b in zip(maps_a, maps_b):
        labels_a = np.asarray(map_a.label_array)
        labels_b = np.asarray(map_b.label_array)
        roi = labels_a > 0
        # Same patient, same ROI: the two maps must cover the same voxels.
        assert np.array_equal(roi, labels_b > 0)
        scores.append(float(adjusted_rand_score(labels_a[roi], labels_b[roi])))
    return scores

## k-means: which K does each criterion pick?
One fitter per criterion, each with the reference settings (2..10
habitats, 10 restarts, seed 0). A candidate K is fitted with the
same seed whatever the criterion, so all curves describe the same
k-means solutions; only the rule that reads them changes.



In [ ]:
KMEANS_CRITERIA = ("elbow", "silhouette", "calinski_harabasz", "davies_bouldin", "gap")
kmeans_k: Dict[str, int] = {}
for criterion in KMEANS_CRITERIA:
    fitter = KMeansHabitatModelFitter(min_habitats=2, max_habitats=10, validation=criterion, n_init=10)
    fitter.set_random_state(0)
    kmeans_k[criterion] = fitter.fit(units, cohort=train).n_habitats
print("k-means, K per criterion:", kmeans_k)

# A list of criteria is a vote. Its report carries every curve; the
# figure marks, on each curve, the K that criterion picked on its own.
voting = KMeansHabitatModelFitter(min_habitats=2, max_habitats=10, validation=list(KMEANS_CRITERIA), n_init=10)
voting.set_random_state(0)
voted_model = voting.fit(units, cohort=train)
print("k-means, K by vote of all five:", voted_model.n_habitats)
report = dict(voted_model.preprocessing_state["selection_report"])
report["selected"] = kmeans_k
fig = plot_cluster_validation_from_report(report, title="k-means: validation criteria")
fig.savefig("out/clustering_choice_kmeans_criteria.png", dpi=150, bbox_inches="tight")
plt.show()

## Gaussian mixture: which K does each criterion pick?
The same rows with ``GmmHabitatModelFitter`` (full covariance, the
default; ``n_init=10`` as for k-means instead of the default 50, to
keep the page fast). Its own default criterion is ``bic``.



In [ ]:
GMM_CRITERIA = ("bic", "bic_elbow", "aic", "silhouette", "calinski_harabasz", "davies_bouldin", "gap")
gmm_k: Dict[str, int] = {}
gmm_models = {}
for criterion in GMM_CRITERIA:
    fitter = GmmHabitatModelFitter(min_habitats=2, max_habitats=10, validation=criterion, n_init=10)
    fitter.set_random_state(0)
    gmm_models[criterion] = fitter.fit(units, cohort=train)
    gmm_k[criterion] = gmm_models[criterion].n_habitats
print("Gaussian mixture, K per criterion:", gmm_k)

voting = GmmHabitatModelFitter(min_habitats=2, max_habitats=10, validation=list(GMM_CRITERIA), n_init=10)
voting.set_random_state(0)
report = dict(voting.fit(units, cohort=train).preprocessing_state["selection_report"])
report["selected"] = gmm_k
fig = plot_cluster_validation_from_report(report, title="Gaussian mixture: validation criteria")
fig.savefig("out/clustering_choice_gmm_criteria.png", dpi=150, bbox_inches="tight")
plt.show()

## Same K, different algorithm
When the two algorithms end up with the same K, are the habitats the
same regions? Compare the reference k-means map with the Gaussian
mixture chosen by ``bic_elbow``, if that picked the same K.



In [ ]:
gmm_same_k = [c for c in GMM_CRITERIA if gmm_k[c] == reference_model.n_habitats]
print("Gaussian-mixture criteria that picked the reference K:", gmm_same_k)
if gmm_same_k:
    criterion = gmm_same_k[0]
    gmm_maps = [gmm_models[criterion].assigner("nearest_centroid")(one) for one in units]
    print(f"ARI k-means elbow vs Gaussian mixture {criterion}:", [round(s, 3) for s in ari(reference_maps, gmm_maps)])
    subject = train[0]
    for name, one_map in (("k-means, elbow", reference_maps[0]), (f"Gaussian mixture, {criterion}", gmm_maps[0])):
        fig = plot_habitat_overlay(
            subject.image("LAP"),
            one_map,
            title=f"{subject.subject_id}: {name} (K = {reference_model.n_habitats})",
            crop_to="labels",
        )
        stem = "kmeans" if name.startswith("k-means") else "gmm"
        fig.savefig(f"out/clustering_choice_{stem}_map.png", dpi=150, bbox_inches="tight")
        plt.show()

## Seed stability
``random_seed`` seeds the k-means supervoxels AND the k-means fit. A
robust definition should give similar maps for any seed. Rerun the
whole Spec with seeds 1 and 2 and compare with seed 0.



In [ ]:
seed_rows = {0: (result.habitat_model.n_habitats, [1.0, 1.0])}
for seed in (1, 2):
    other = Study(make_spec(seed)).fit_predict(train)
    seed_rows[seed] = (other.habitat_model.n_habitats, ari(result.habitat_maps, other.habitat_maps))
for seed, (k, scores) in seed_rows.items():
    print(f"seed {seed}: K = {k}, ARI vs seed 0 per patient = {[round(s, 3) for s in scores]}")

## How to read the result
On the build machine, with these two demo patients (60 pooled
supervoxels):

* k-means: ``elbow`` and ``davies_bouldin`` picked K = 4,
  ``silhouette`` and ``calinski_harabasz`` picked K = 2, ``gap`` picked
  K = 10 (the top of the range). The vote of all five picked K = 2.
* Gaussian mixture: ``bic_elbow`` picked K = 4, ``bic`` and ``aic``
  picked K = 10 (the top of the range), the label-based scores picked
  K = 2, ``gap`` picked K = 7.
* The criteria disagree. That is normal: ``silhouette`` and
  ``calinski_harabasz`` reward a few well separated groups and often
  prefer 2; ``gap``, ``bic`` and ``aic`` keep improving as K grows on
  so few rows, so they run to the end of the range. A pick at the edge
  of the range means "the criterion did not find an optimum here", not
  "K = 10 is right".
* The seed-stability lines above show whether the K and the maps
  survive a different random start on these data.

Guidance for a study:

* Choose the criterion (and the K range) before looking at outcomes
  and report it; ``describe_methods()`` records it with the spec.
* Prefer a criterion whose pick is inside the range; widen the range
  or change the criterion if it sits on the edge.
* Show the curves (the figures above) in the supplement, report the
  seed stability, and, if habitats are interpreted biologically,
  check that neighbouring Ks tell the same story.



## Where to go next
* The supervoxel step before the fit:
  :doc:`/auto_examples/03_clustering/plot_01_supervoxel_method_and_count`.
* Fitting and assigning on their own pages:
  :doc:`/auto_examples/07_advanced/plot_05_fit`,
  :doc:`/auto_examples/07_advanced/plot_06_assign`.
* Matching habitat numbers between two fits:
  :doc:`/auto_examples/05_validation_and_reuse/plot_02_matching_labels`.
* Cleaning the final maps:
  :doc:`/auto_examples/01_building_habitat_maps/plot_08_cleaning_maps`.

